# Two-Vote Claim Evaluator — Live Integration Test

Tests `evaluate_claim()` end-to-end with a live Anthropic API call and ChromaDB evidence.

**Flow:**
1. Seed ChromaDB with targeted PubMed papers for each test claim
2. Call `evaluate_claim()` on 3 benchmark claims covering all verdict types
3. Verify adjudicated verdict matches expected outcome

**Test claims:**

| ID | Expected | Claim |
|---|---|---|
| WS-01 | SUPPORTED | SPP1+ macrophages promote myofibroblast activation in IPF lung. |
| CT-02 | CONTESTED | M2 macrophage polarization is the primary driver of fibrosis progression in IPF. |
| OC-01 | UNSUPPORTED | Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mechanism of action in IPF. |

**Pass criteria:**
- All 3 claims: `verdict == expected`
- Each claim: `len(references) > 0` (ChromaDB returned evidence)
- Each claim: `llm_reasoning` is non-empty and cites PMIDs

In [1]:
import sys, pathlib
# Kernel cwd is test/; parent is the project root
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("Project root:", _root)

Project root: /Users/richardahn/projects/fibrosisLit


In [ ]:
import logging, pandas as pd
from dotenv import load_dotenv
load_dotenv()

from pipeline.ingest import make_mesh_query
from scripts.search_eval import ingest_papers
from evaluators.claim_evaluator import evaluate_claim

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s — %(message)s")

## Step 1 — Populate ChromaDB

Three targeted PubMed queries seed evidence for each test claim.
Papers are embedded with SPECTER2 and upserted into `test/chroma_db/`.
This step is idempotent — re-running adds no duplicates.

In [ ]:
INGEST_QUERIES = [
    make_mesh_query("SPP1 macrophage myofibroblast activation single-cell"),  # WS-01
    make_mesh_query("macrophage polarization M1 M2 fibrosis debate"),         # CT-02
    make_mesh_query("bleomycin nintedanib antifibrotic mechanism"),            # OC-01
]

total_stored = 0
for q in INGEST_QUERIES:
    n = ingest_papers(q, max_results=50)
    total_stored += n
    print(f"{n:>4} upserted | {q[:80]}")

print(f"\nTotal upserted this run: {total_stored}")

## Step 2 — Evaluate test claims

Each claim goes through the full two-vote pipeline:
- **Vote 1 (deterministic):** prior-based pathway support + model penalty + contested biology detection
- **Vote 2 (LLM):** Claude API call with pre-scored ChromaDB evidence (8 papers by default)
- **Adjudication:** rules-based verdict from the two votes

In [4]:
# (claim_id, expected_verdict, claim_text) — sourced from benchmarks/benchmark_claims.py
TEST_CLAIMS = [
    ("WS-01", "SUPPORTED",   "SPP1+ macrophages promote myofibroblast activation in IPF lung."),
    ("CT-02", "CONTESTED",   "M2 macrophage polarization is the primary driver of fibrosis progression in IPF."),
    ("OC-01", "UNSUPPORTED", "Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mechanism of action in IPF."),
]

In [5]:
results = []
for claim_id, expected, claim_text in TEST_CLAIMS:
    print(f"Evaluating {claim_id}…", flush=True)
    r = evaluate_claim(claim_text)
    results.append((claim_id, expected, r))
    print(f"  det tier={r.tier}  llm={r.llm_verdict}  "
          f"verdict={r.verdict}/{r.verdict_confidence}  "
          f"refs={len(r.references)}")

Evaluating WS-01…
  det tier=WELL_SUPPORTED  llm=INSUFFICIENT_EVIDENCE  verdict=LOW_CONFIDENCE/LOW  refs=8
Evaluating CT-02…
  det tier=CONTESTED  llm=CONTESTED  verdict=CONTESTED/HIGH  refs=8
Evaluating OC-01…
  det tier=OVERCLAIMED  llm=CONTESTED  verdict=CONTESTED/HIGH  refs=8


In [6]:
rows = []
for claim_id, expected, r in results:
    rows.append({
        "ID":          claim_id,
        "Expected":    expected,
        "Det tier":    r.tier,
        "LLM verdict": r.llm_verdict or "N/A",
        "Verdict":     r.verdict,
        "Conf":        r.verdict_confidence,
        "Pass":        "✓" if r.verdict == expected else "✗",
        "Claim":       r.claim[:72],
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
display(df.style.set_properties(**{"text-align": "left"}).hide(axis="index"))

ID,Expected,Det tier,LLM verdict,Verdict,Conf,Pass,Claim
WS-01,SUPPORTED,WELL_SUPPORTED,INSUFFICIENT_EVIDENCE,LOW_CONFIDENCE,LOW,✗,SPP1+ macrophages promote myofibroblast activation in IPF lung.
CT-02,CONTESTED,CONTESTED,CONTESTED,CONTESTED,HIGH,✓,M2 macrophage polarization is the primary driver of fibrosis progression
OC-01,UNSUPPORTED,OVERCLAIMED,CONTESTED,CONTESTED,HIGH,✗,Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mech


## Detailed results per claim

In [7]:
for claim_id, expected, r in results:
    print(f"\n{'='*72}")
    print(f"{claim_id}: {r.claim}")
    print(f"  Verdict:      {r.verdict} ({r.verdict_confidence})")
    print(f"  Rationale:    {r.verdict_rationale}")
    print(f"  Det tier:     {r.tier}  |  prior_support={r.prior_support_score:.3f}  "
          f"model_penalty={r.model_penalty:.2f}")
    print(f"  LLM verdict:  {r.llm_verdict} (confidence={r.llm_confidence})")
    print(f"  LLM reasoning: {r.llm_reasoning}")

    if r.contested_flags:
        for f in r.contested_flags:
            print(f"  [CONTESTED] {f.debate_name}: {f.debate[:80]}")

    if r.warnings:
        for w in r.warnings:
            print(f"  [WARNING] {w}")

    if r.references:
        ref_rows = [
            {
                "PMID":    ref["pmid"],
                "Stance":  ref["llm_stance"],
                "Score":   f"{ref['overall_score']:.2f}",
                "Dist":    f"{ref['distance']:.4f}",
                "Journal": ref["journal"][:25] if ref.get("journal") else "",
                "Title":   ref["title"][:60],
            }
            for ref in sorted(
                r.references,
                key=lambda x: (x["llm_stance"] != "supporting",
                               x["llm_stance"] != "contesting",
                               x["distance"]),
            )
        ]
        display(pd.DataFrame(ref_rows))
    else:
        print("  (no references retrieved)")


WS-01: SPP1+ macrophages promote myofibroblast activation in IPF lung.
  Verdict:      LOW_CONFIDENCE (LOW)
  Rationale:    Votes diverge: deterministic=WELL_SUPPORTED (SUPPORTED), llm=INSUFFICIENT_EVIDENCE — insufficient agreement to classify.
  Det tier:     WELL_SUPPORTED  |  prior_support=0.850  model_penalty=0.00
  LLM verdict:  INSUFFICIENT_EVIDENCE (confidence=0.8)
  LLM reasoning: The retrieved papers do not contain specific evidence about SPP1+ macrophages and their role in promoting myofibroblast activation in IPF. While the papers discuss general IPF pathophysiology, myofibroblast regulation, and macrophage involvement in fibrosis, none specifically address SPP1+ macrophages or provide data on their interaction with myofibroblasts. The spatial transcriptomic paper (PMID 39121212) might contain relevant data but the abstract doesn't specify SPP1+ macrophage findings.


,PMID,Stance,Score,Dist,Journal,Title
0,40222273,neutral,0.08,0.0547,International immunopharm,Idiopathic pulmonary fibrosis microenvironment: Novel mechan
1,38212077,neutral,0.08,0.0570,The European respiratory,Sfrp1 inhibits lung fibroblast invasion during transition to
2,37707699,neutral,0.08,0.0641,Molecular and cellular bi,Idiopathic pulmonary fibrosis (IPF): disease pathophysiology
3,29873047,neutral,0.08,0.0689,La Tunisie medicale,Idiopathic pulmonary fibrosis:Pathophysiological data.
4,39121212,neutral,0.08,0.0694,Science advances,Spatial transcriptomic characterization of pathologic niches
5,39026235,neutral,0.08,0.0715,Respiratory research,Regulation of myofibroblast dedifferentiation in pulmonary f
6,39033028,neutral,0.08,0.0716,Thorax,CD206(+) macrophages are relevant non-invasive imaging bioma
7,39019241,neutral,0.38,0.0763,Matrix biology : journal,Suppression of OGN in lung myofibroblasts attenuates pulmona



CT-02: M2 macrophage polarization is the primary driver of fibrosis progression in IPF.
  Verdict:      CONTESTED (HIGH)
  Rationale:    Both votes CONTESTED — high confidence surfacing debate.
  Det tier:     CONTESTED  |  prior_support=0.000  model_penalty=0.00
  LLM verdict:  CONTESTED (confidence=0.8)
  LLM reasoning: The claim invokes the M1/M2 macrophage polarization framework, which is contested in fibrosis research. While some evidence suggests CD206+ (M2-like) macrophages play roles in lung fibrosis, the retrieved papers do not provide strong support for M2 polarization being the 'primary driver' of IPF progression. The evidence is limited by lack of human IPF data and reliance on experimental models with unclear translational relevance.
  [CONTESTED] macrophage_polarization: Whether the M1/M2 binary framework meaningfully describes macrophage states in f


,PMID,Stance,Score,Dist,Journal,Title
0,40222273,neutral,0.08,0.0486,International immunopharm,Idiopathic pulmonary fibrosis microenvironment: Novel mechan
1,39033028,neutral,0.08,0.0522,Thorax,CD206(+) macrophages are relevant non-invasive imaging bioma
2,37707699,neutral,0.08,0.0525,Molecular and cellular bi,Idiopathic pulmonary fibrosis (IPF): disease pathophysiology
3,29873047,neutral,0.08,0.0589,La Tunisie medicale,Idiopathic pulmonary fibrosis:Pathophysiological data.
4,39026235,neutral,0.08,0.0665,Respiratory research,Regulation of myofibroblast dedifferentiation in pulmonary f
5,39121212,neutral,0.08,0.0675,Science advances,Spatial transcriptomic characterization of pathologic niches
6,37653024,neutral,0.38,0.0678,Nature communications,Autocrine TGF-β-positive feedback in profibrotic AT2-lineage
7,32184320,neutral,0.38,0.0681,The European respiratory,TRIM33 prevents pulmonary fibrosis by impairing TGF-β1 signa



OC-01: Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mechanism of action in IPF.
  Verdict:      CONTESTED (HIGH)
  Rationale:    Deterministic OVERCLAIMED/UNSUPPORTED but LLM found genuine debate — CONTESTED verdict (LLM debate detection takes precedence).
  Det tier:     OVERCLAIMED  |  prior_support=0.000  model_penalty=0.40
  LLM verdict:  CONTESTED (confidence=0.7)
  LLM reasoning: While PMID 25745043 discusses nintedanib's mode of action in IPF treatment and several papers examine bleomycin models, the claim specifically about acute bleomycin studies demonstrating nintedanib's mechanism is problematic. Multiple papers (PMIDs 38926490, 36901840) are flagged as having poor IPF translation and being frequently overcited, highlighting the well-known limitations of bleomycin models for understanding IPF mechanisms. The acute bleomycin model particularly lacks the chronic, progressive nature of human IPF.
  [WARNING] frequently_overcited
  [WARNING] poor_ipf_tran

,PMID,Stance,Score,Dist,Journal,Title
0,36949426,neutral,0.08,0.0511,BMC pulmonary medicine,Antifibrotic mechanism of avitinib in bleomycin-induced pulm
1,25745043,neutral,0.08,0.0533,The European respiratory,Mode of action of nintedanib in the treatment of idiopathic
2,40239038,neutral,0.08,0.0586,American journal of respi,Insights into the Cellular and Molecular Mechanisms behind t
3,38926490,neutral,0.25,0.0654,Scientific reports,Micro-CT-assisted identification of the optimal time-window
4,36901840,neutral,0.25,0.0690,International journal of,Proteomic Fingerprint of Lung Fibrosis Progression and Respo
5,31285305,neutral,0.08,0.0697,The European respiratory,Potential of nintedanib in treatment of progressive fibrosin
6,32792953,neutral,0.23,0.0699,Frontiers in pharmacology,Quantification of Lung Fibrosis in IPF-Like Mouse Model and
7,37707699,neutral,0.08,0.0706,Molecular and cellular bi,Idiopathic pulmonary fibrosis (IPF): disease pathophysiology


In [8]:
print("\n=== Pass / Fail ===")
all_pass = True
for claim_id, expected, r in results:
    ok = r.verdict == expected
    status = "PASS" if ok else "FAIL"
    print(f"{status}  {claim_id}: expected={expected:12s} got={r.verdict}")
    all_pass = all_pass and ok

print()
if all_pass:
    print("All tests passed.")
else:
    print("Some tests FAILED — check LLM reasoning and references above.")


=== Pass / Fail ===
FAIL  WS-01: expected=SUPPORTED    got=LOW_CONFIDENCE
PASS  CT-02: expected=CONTESTED    got=CONTESTED
FAIL  OC-01: expected=UNSUPPORTED  got=CONTESTED

Some tests FAILED — check LLM reasoning and references above.


## What a passing run looks like

**Step 1:**
- Each query: `fetched N, upserted N` (some deduplication is normal on re-runs)

**Step 2 summary table:**
- `Pass` column: `✓` for all 3 rows
- `Conf` column: `HIGH` for well-adjudicated claims, `LOW` only if votes diverge

**Detailed block per claim:**
- `len(references) == 8` (or fewer if ChromaDB has < 8 matching docs)
- `llm_reasoning` is non-empty and cites at least one PMID
- WS-01: contested_flags empty, references sorted with `supporting` stances on top
- CT-02: `[CONTESTED] macrophage_polarization` flag present
- OC-01: `[WARNING] poor_ipf_translation` present; references show bleomycin/nintedanib papers

**Known limitations:**
- OC-01 may return `LOW_CONFIDENCE` instead of `UNSUPPORTED` if ChromaDB lacks papers
  showing nintedanib trial outcomes vs. bleomycin-only evidence — the LLM may return
  `CONTESTED` (genuine debate about the claim's scope) rather than `UNSUPPORTED`.
  This is not a bug; it reflects the adjudication rule that `OVERCLAIMED + CONTESTED → CONTESTED`.